# Module 3 - Data Project

This version of Matching has a different data set than Matchit and the previous versions of Matching, all called lalonde.
See https://github.com/vlaskinvlad/coursera-causality-crash-course/blob/master/causality_w3.ipynb

In [36]:
#install packages
install.packages("tableone")
install.packages("Matching")
library(Matching)
library(tableone)

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [14]:
# two versions of lalonde data, get the right one
data(lalonde)
View(lalonde)

,age,educ,black,hisp,married,nodegr,re74,re75,re78,u74,u75,treat
,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>
1,37,11,1,0,1,1,0,0,9930.05,1,1,1
2,22,9,0,1,0,1,0,0,3595.89,1,1,1
3,30,12,1,0,0,0,0,0,24909.50,1,1,1
4,27,11,1,0,0,1,0,0,7506.15,1,1,1
5,33,8,1,0,0,1,0,0,289.79,1,1,1
6,22,9,1,0,0,1,0,0,4056.49,1,1,1
7,23,12,1,0,0,0,0,0,0.00,1,1,1
8,32,11,1,0,0,1,0,0,8472.16,1,1,1
9,22,16,1,0,0,0,0,0,2164.02,1,1,1


In [15]:
xvars<-c("age","educ","black", "hisp","married","nodegr","re74","re75", "re78")
matchedtab1<-CreateTableOne(vars=xvars, strata ="treat", 
                            data=lalonde, test = FALSE)
print(matchedtab1, smd = TRUE)

                     Stratified by treat
                      0                 1                 SMD   
  n                       260               185                 
  age (mean (SD))       25.05 (7.06)      25.82 (7.16)     0.107
  educ (mean (SD))      10.09 (1.61)      10.35 (2.01)     0.141
  black (mean (SD))      0.83 (0.38)       0.84 (0.36)     0.044
  hisp (mean (SD))       0.11 (0.31)       0.06 (0.24)     0.175
  married (mean (SD))    0.15 (0.36)       0.19 (0.39)     0.094
  nodegr (mean (SD))     0.83 (0.37)       0.71 (0.46)     0.304
  re74 (mean (SD))    2107.03 (5687.91) 2095.57 (4886.62)  0.002
  re75 (mean (SD))    1266.91 (3102.98) 1532.06 (3219.25)  0.084
  re78 (mean (SD))    4554.80 (5483.84) 6349.15 (7867.40)  0.265


In [28]:
# Fit a model on the confounders only, not the outcome value re78
ps2model<-glm(treat~age+educ+black+hisp+married+nodegr+re74+re75,
    family=binomial(link="logit"),data=lalonde)

#show coefficients etc
summary(ps2model)
#create propensity score
ps2score<-ps2model$fitted.values


Call:
glm(formula = treat ~ age + educ + black + hisp + married + nodegr + 
    re74 + re75, family = binomial(link = "logit"), data = lalonde)

Coefficients:
              Estimate Std. Error z value Pr(>|z|)   
(Intercept)  1.178e+00  1.056e+00   1.115  0.26474   
age          4.698e-03  1.433e-02   0.328  0.74297   
educ        -7.124e-02  7.173e-02  -0.993  0.32061   
black       -2.247e-01  3.655e-01  -0.615  0.53874   
hisp        -8.528e-01  5.066e-01  -1.683  0.09228 . 
married      1.636e-01  2.769e-01   0.591  0.55463   
nodegr      -9.035e-01  3.135e-01  -2.882  0.00395 **
re74        -3.161e-05  2.584e-05  -1.223  0.22122   
re75         6.161e-05  4.358e-05   1.414  0.15744   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 604.20  on 444  degrees of freedom
Residual deviance: 587.22  on 436  degrees of freedom
AIC: 605.22

Number of Fisher Scoring iterations: 4


In [33]:
set.seed(931139)
# do greedy matching on logit(PS) using Match without a caliper

ps2match<-Match(Y=lalonde$re78, Tr=lalonde$treat,M=1,X=ps2score)
summary(ps2match)


Estimate...  2624.3 
AI SE......  802.19 
T-stat.....  3.2714 
p.val......  0.0010702 

Original number of observations..............  445 
Original number of treated obs...............  185 
Matched number of observations...............  185 
Matched number of observations  (unweighted).  344 



In [34]:
matched2<-lalonde[unlist(ps2match[c("index.treated","index.control")]),]
# get standardized differences
matchedtab2<-CreateTableOne(vars=xvars, strata ="treat", 
                            data=matched2, test = FALSE)
print(matchedtab2, smd = TRUE)

                     Stratified by treat
                      0                 1                 SMD   
  n                       344               344                 
  age (mean (SD))       24.47 (6.47)      24.62 (6.90)     0.023
  educ (mean (SD))      10.19 (1.49)      10.37 (1.64)     0.115
  black (mean (SD))      0.92 (0.28)       0.89 (0.31)     0.088
  hisp (mean (SD))       0.03 (0.17)       0.04 (0.20)     0.063
  married (mean (SD))    0.12 (0.33)       0.15 (0.36)     0.076
  nodegr (mean (SD))     0.83 (0.38)       0.78 (0.41)     0.110
  re74 (mean (SD))    1295.89 (4302.81) 1780.12 (4193.15)  0.114
  re75 (mean (SD))    1320.29 (3463.59) 1132.30 (2549.89)  0.062
  re78 (mean (SD))    3624.86 (4432.07) 5600.71 (7086.87)  0.334


In [19]:
set.seed(931139)
# do greedy matching on logit(PS) using Match with a caliper

ps2match<-Match(Y=lalonde$re78, Tr=lalonde$treat,M=1,X=ps2score, caliper=0.1)
summary(ps2match)


Estimate...  2561.6 
AI SE......  783.34 
T-stat.....  3.2701 
p.val......  0.0010752 

Original number of observations..............  445 
Original number of treated obs...............  185 
Matched number of observations...............  181 
Matched number of observations  (unweighted).  340 

Caliper (SDs)........................................   0.1 
Number of obs dropped by 'exact' or 'caliper'  4 



In [21]:
matched2<-lalonde[unlist(ps2match[c("index.treated","index.control")]),]
#outcome analysis
y_trt<-matched2$re78[matched2$treat==1]
y_con<-matched2$re78[matched2$treat==0]

#pairwise difference
diffy<-y_trt-y_con

print(mean(diffy))

#paired t-test
t.test(diffy)


[1] 1934.842
[1] 1934.842



	One Sample t-test

data:  diffy
t = 4.3535, df = 339, p-value = 1.777e-05
alternative hypothesis: true mean is not equal to 0
95 percent confidence interval:
 1060.640 2809.045
sample estimates:
mean of x 
 1934.842 


In [38]:
# write.csv(lalonde, "~/work/lalonde-matching.csv", row.names=FALSE)